In [0]:
use catalog identifier(:catalog)

In [0]:
-- Create the ops.user_entitlements table that drives both the mask and the row filter
CREATE SCHEMA IF NOT EXISTS ops;

CREATE OR REPLACE TABLE ops.user_entitlements (
  user_email    STRING,
  allowed_city  STRING,
  can_view_pii  BOOLEAN
);

In [0]:
-- G1: Grant the built-in group 'account users' only:
--   USE CATALOG on this catalog
--   USE SCHEMA + SELECT on schema gold
-- Grant nothing else to anyone.
GRANT USE CATALOG ON CATALOG identifier(:catalog) TO `account users`;

-- Schema grants use the current catalog context (set by USE CATALOG above)
GRANT USE SCHEMA ON SCHEMA gold TO `account users`;
GRANT SELECT      ON SCHEMA gold TO `account users`;

In [0]:
-- G2: Mask phone in gold.dim_member
-- Real value if can_view_pii = TRUE for the current user, otherwise XXXXXX + last 4 digits
CREATE OR REPLACE FUNCTION ops.mask_phone(phone_value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN (SELECT can_view_pii FROM ops.user_entitlements WHERE user_email = current_user()) = TRUE
    THEN phone_value
    ELSE 'XXXXXX' || RIGHT(phone_value, 4)
  END;

-- Apply the mask to the phone column (idempotent: re-run after gold rebuild)
ALTER TABLE gold.dim_member ALTER COLUMN phone SET MASK ops.mask_phone;

In [0]:
-- G3: Row filter on gold.agg_daily_station
-- Show rows whose city = allowed_city, or all rows if allowed_city = 'ALL'
-- No entitlement row → no rows (EXISTS returns FALSE)
CREATE OR REPLACE FUNCTION ops.filter_city(city STRING)
RETURNS BOOLEAN
RETURN EXISTS(
  SELECT 1 FROM ops.user_entitlements
  WHERE user_email = current_user()
    AND (allowed_city = 'ALL' OR allowed_city = city)
);

-- Apply the row filter at the table level (idempotent: re-run after gold rebuild)
ALTER TABLE gold.agg_daily_station SET ROW FILTER ops.filter_city ON (city);

In [0]:
-- G4: Evidence collection
-- Step 1: Set entitlement to Jaipur, can_view_pii = FALSE
DELETE FROM ops.user_entitlements WHERE user_email = current_user();
INSERT INTO ops.user_entitlements (user_email, allowed_city, can_view_pii)
VALUES (current_user(), 'Jaipur', FALSE);

-- Step 2: Capture evidence
-- ROW_FILTER_ROW_COUNT_JAIPUR: rows visible in gold.agg_daily_station (row filter applied)
CREATE OR REPLACE TABLE ops.governance_evidence (
  evidence_name   STRING,
  observed_value  STRING
);

INSERT INTO ops.governance_evidence (evidence_name, observed_value)
SELECT 'ROW_FILTER_ROW_COUNT_JAIPUR', CAST(COUNT(*) AS STRING)
FROM gold.agg_daily_station;

-- MASKED_PHONE_M0001: current phone of M0001 (mask applied)
INSERT INTO ops.governance_evidence (evidence_name, observed_value)
SELECT 'MASKED_PHONE_M0001', phone
FROM gold.dim_member
WHERE member_id = 'M0001';

-- Step 3: Set entitlement to ALL, can_view_pii = FALSE
UPDATE ops.user_entitlements
SET allowed_city = 'ALL', can_view_pii = FALSE
WHERE user_email = current_user();

In [0]:
-- Verify mask and filter are still in place after Run 3
-- Re-apply if the gold tables were rebuilt (CREATE OR REPLACE TABLE drops masks/filters)
ALTER TABLE gold.dim_member ALTER COLUMN phone SET MASK ops.mask_phone;
ALTER TABLE gold.agg_daily_station SET ROW FILTER ops.filter_city ON (city);

-- Show current evidence
SELECT * FROM ops.governance_evidence;

-- Show current entitlement
SELECT * FROM ops.user_entitlements;

-- Verify row filter: should see ALL cities now (allowed_city = 'ALL')
SELECT DISTINCT city FROM gold.agg_daily_station ORDER BY city;

-- Verify mask: M0001 phone should still be masked (can_view_pii = FALSE)
SELECT member_id, phone FROM gold.dim_member WHERE member_id = 'M0001';

In [0]:

COMMENT ON CATALOG :catalog IS
  'Voltpath EV charging practice paper catalog. Contains landing, bronze, silver, gold, and ops schemas for the full data engineering and analytics pipeline.';



COMMENT ON SCHEMA landing IS
  'Raw landing zone for ingested source files (CSV/JSON) before any transformation. Contains the raw volume.';
COMMENT ON SCHEMA bronze IS
  'Bronze layer: raw data loaded from landing into Delta tables with minimal cleansing. First persisted layer in the medallion architecture.';
COMMENT ON SCHEMA silver IS
  'Silver layer: cleaned, deduplicated, and conformed data ready for analytics. Includes quarantined records and exploded fault data.';
COMMENT ON SCHEMA gold IS
  'Gold layer: business-ready dimensional model (star schema) and pre-aggregated tables for reporting, dashboards, and governance.';
COMMENT ON SCHEMA ops IS
  'Operational schema: pipeline metadata, reconciliation results, governance evidence, and audit tables.';
  

COMMENT ON VOLUME landing.raw IS
  'Managed volume for raw source files (CSV/JSON) uploaded before bronze ingestion.';
COMMENT ON VOLUME ops.pipeline_state IS
  'Managed volume storing pipeline state and checkpoint files for incremental processing.';


COMMENT ON TABLE bronze.chargers IS
  'Raw charger master data ingested from source files. Unvalidated, as-is from landing.';
COMMENT ON TABLE bronze.members IS
  'Raw member master data ingested from source files. Unvalidated, as-is from landing.';
COMMENT ON TABLE bronze.sessions IS
  'Raw EV charging session records ingested from source files. Unvalidated, as-is from landing.';
COMMENT ON TABLE bronze.stations IS
  'Raw station master data ingested from source files. Unvalidated, as-is from landing.';
COMMENT ON TABLE bronze.tariffs IS
  'Raw tariff and plan data ingested from source files. Unvalidated, as-is from landing.';


COMMENT ON TABLE silver.chargers IS
  'Cleansed and conformed charger dimension data. Deduplicated and type-cast from bronze.';
COMMENT ON TABLE silver.members IS
  'Cleansed and conformed member dimension data with deduplication. Source for gold.dim_member SCD Type 2.';
COMMENT ON TABLE silver.sessions IS
  'Cleansed and conformed charging session records. Validated, deduplicated, and enriched with derived columns.';
COMMENT ON TABLE silver.session_faults IS
  'Exploded fault records extracted from session data. One row per fault per session.';
COMMENT ON TABLE silver.quarantine_sessions IS
  'Sessions that failed validation rules and were quarantined for manual review.';
COMMENT ON TABLE silver.stations IS
  'Cleansed and conformed station dimension data. Deduplicated and type-cast from bronze.';
COMMENT ON TABLE silver.tariffs IS
  'Cleansed and conformed tariff and plan data. Deduplicated and type-cast from bronze.';

COMMENT ON TABLE gold.agg_daily_station IS
  'Pre-aggregated daily station-level metrics for dashboards and reporting. One row per station per day.';
COMMENT ON TABLE gold.dim_charger IS
  'Charger dimension table with station city denormalised for convenient joins.';
COMMENT ON TABLE gold.dim_date IS
  'Date dimension table with calendar attributes (day name, ISO week, weekend flag) for time-based analysis.';
COMMENT ON TABLE gold.dim_member IS
  'Member dimension table (SCD Type 2) tracking plan history over time. Surrogate key member_sk links to fact_sessions.';
COMMENT ON TABLE gold.dim_stations IS
  'Station dimension table with site type and opening date.';
COMMENT ON TABLE gold.fact_sessions IS
  'Session-level fact table with billing breakdown, plan lookup, and fault counts. Star schema centre table.';
COMMENT ON TABLE gold.v_kpi_summary IS
  'Summary view of key performance indicators: completed/failed sessions, success rate, energy, revenue leakage, and active members.';


COMMENT ON TABLE ops.charger_changes_batch2 IS
  'Audit table tracking charger configuration changes from batch 2 of the pipeline.';
COMMENT ON TABLE ops.checkpoint_counts IS
  'Row counts at each pipeline checkpoint for reconciliation and data quality monitoring.';
COMMENT ON TABLE ops.governance_evidence IS
  'Evidence table capturing governance rule verification results (row filter and column mask checks).';
COMMENT ON TABLE ops.reconciliation_results IS
  'Results of silver-to-gold reconciliation checks comparing row counts and data integrity.';
COMMENT ON TABLE ops.sessions_drill IS
  'Drill-down detail table for session-level investigation and root-cause analysis.';
COMMENT ON TABLE ops.time_travel_check IS
  'Time travel verification results for Delta table version history and audit trail.';
COMMENT ON TABLE ops.user_entitlements IS
  'User entitlement table driving row filters (allowed_city) and column masks (can_view_pii) via governance functions.';


COMMENT ON COLUMN gold.agg_daily_station.session_date IS 'Calendar date of the aggregation period.';
COMMENT ON COLUMN gold.agg_daily_station.station_id IS 'Unique identifier for the charging station. Foreign key to dim_stations.';
COMMENT ON COLUMN gold.agg_daily_station.city IS 'City where the station is located.';
COMMENT ON COLUMN gold.agg_daily_station.completed_sessions IS 'Count of sessions with status COMPLETED on this date at this station.';
COMMENT ON COLUMN gold.agg_daily_station.failed_sessions IS 'Count of sessions with non-COMPLETED status on this date at this station.';
COMMENT ON COLUMN gold.agg_daily_station.energy_kwh IS 'Total energy delivered in kWh for completed sessions.';
COMMENT ON COLUMN gold.agg_daily_station.billed_amount IS 'Total amount billed to members for completed sessions.';
COMMENT ON COLUMN gold.agg_daily_station.amount_paid IS 'Total amount actually paid by members.';
COMMENT ON COLUMN gold.agg_daily_station.mismatch_sessions IS 'Count of sessions where billed amount differs from amount paid.';
COMMENT ON COLUMN gold.agg_daily_station.fault_count IS 'Total number of faults recorded across all sessions at this station on this date.';
COMMENT ON COLUMN gold.agg_daily_station.utilisation_pct IS 'Station utilisation percentage for the day.';


COMMENT ON COLUMN gold.dim_charger.charger_id IS 'Unique identifier for the charger.';
COMMENT ON COLUMN gold.dim_charger.station_id IS 'Foreign key to dim_stations identifying the parent station.';
COMMENT ON COLUMN gold.dim_charger.connector_type IS 'Physical connector type (e.g., CCS, CHAdeMO, Type 2).';
COMMENT ON COLUMN gold.dim_charger.max_kw IS 'Maximum power output of the charger in kilowatts.';
COMMENT ON COLUMN gold.dim_charger.is_active IS 'Whether the charger is currently operational.';
COMMENT ON COLUMN gold.dim_charger.commissioned_on IS 'Date the charger was put into service.';
COMMENT ON COLUMN gold.dim_charger.station_city IS 'City of the parent station, denormalised for convenience.';

COMMENT ON COLUMN gold.dim_date.date_key IS 'Integer surrogate key in YYYYMMDD format.';
COMMENT ON COLUMN gold.dim_date.calendar_date IS 'The actual calendar date.';
COMMENT ON COLUMN gold.dim_date.day_name IS 'Name of the day of the week (e.g., Monday, Tuesday).';
COMMENT ON COLUMN gold.dim_date.iso_week IS 'ISO week number (1-53).';
COMMENT ON COLUMN gold.dim_date.is_weekend IS 'TRUE if the date falls on Saturday or Sunday.';


COMMENT ON COLUMN gold.dim_member.member_sk IS 'Surrogate key for the member dimension (SCD Type 2).';
COMMENT ON COLUMN gold.dim_member.member_id IS 'Business key identifying the EV charging member.';
COMMENT ON COLUMN gold.dim_member.full_name IS 'Full name of the member. Contains PII.';
COMMENT ON COLUMN gold.dim_member.email IS 'Email address of the member. Contains PII.';
COMMENT ON COLUMN gold.dim_member.phone IS 'Phone number of the member. Contains PII; masked via governance policy.';
COMMENT ON COLUMN gold.dim_member.city IS 'City of residence of the member.';
COMMENT ON COLUMN gold.dim_member.plan IS 'Membership plan at the current point in time.';
COMMENT ON COLUMN gold.dim_member.joined_on IS 'Date the member registered with Voltpath.';
COMMENT ON COLUMN gold.dim_member.effective_from IS 'SCD Type 2 effective-from date for this member record version.';
COMMENT ON COLUMN gold.dim_member.effective_to IS 'SCD Type 2 effective-to date; NULL for the current version.';
COMMENT ON COLUMN gold.dim_member.is_current IS 'TRUE if this is the current (most recent) version of the member record.';


COMMENT ON COLUMN gold.dim_stations.station_id IS 'Unique identifier for the charging station.';
COMMENT ON COLUMN gold.dim_stations.station_name IS 'Human-readable name of the station.';
COMMENT ON COLUMN gold.dim_stations.city IS 'City where the station is located.';
COMMENT ON COLUMN gold.dim_stations.site_type IS 'Type of site (e.g., highway, urban, retail).';
COMMENT ON COLUMN gold.dim_stations.opened_on IS 'Date the station was opened for public use.';


COMMENT ON COLUMN gold.fact_sessions.session_id IS 'Unique identifier for the charging session.';
COMMENT ON COLUMN gold.fact_sessions.date_key IS 'Foreign key to dim_date for the session date.';
COMMENT ON COLUMN gold.fact_sessions.charger_id IS 'Foreign key to dim_charger identifying the charger used.';
COMMENT ON COLUMN gold.fact_sessions.station_id IS 'Foreign key to dim_stations identifying the station.';
COMMENT ON COLUMN gold.fact_sessions.member_id IS 'Business key identifying the member who initiated the session.';
COMMENT ON COLUMN gold.fact_sessions.status IS 'Session status (e.g., COMPLETED, FAILED, IN_PROGRESS).';
COMMENT ON COLUMN gold.fact_sessions.time_band IS 'Time-of-day band categorisation (e.g., peak, off-peak, night).';
COMMENT ON COLUMN gold.fact_sessions.energy_kwh IS 'Energy delivered during the session in kWh.';
COMMENT ON COLUMN gold.fact_sessions.duration_minutes IS 'Total session duration in minutes.';
COMMENT ON COLUMN gold.fact_sessions.idle_minutes IS 'Minutes the vehicle remained connected after charging completed.';
COMMENT ON COLUMN gold.fact_sessions.amount_paid IS 'Amount actually paid by the member for this session.';
COMMENT ON COLUMN gold.fact_sessions.member_sk IS 'Surrogate key to dim_member for point-in-time plan lookup.';
COMMENT ON COLUMN gold.fact_sessions.plan_at_session IS 'Membership plan effective at the time of the session.';
COMMENT ON COLUMN gold.fact_sessions.rate_per_kwh IS 'Tariff rate per kWh applied to this session.';
COMMENT ON COLUMN gold.fact_sessions.energy_charge IS 'Calculated energy charge before discounts and fees.';
COMMENT ON COLUMN gold.fact_sessions.plan_discount IS 'Discount amount applied based on the member plan.';
COMMENT ON COLUMN gold.fact_sessions.idle_fee IS 'Fee charged for idle time beyond the free threshold.';
COMMENT ON COLUMN gold.fact_sessions.billed_amount IS 'Total amount billed (energy_charge - plan_discount + idle_fee).';
COMMENT ON COLUMN gold.fact_sessions.billing_status IS 'Status of billing reconciliation (e.g., matched, mismatch).';
COMMENT ON COLUMN gold.fact_sessions.fault_count IS 'Number of faults recorded during this session.';


COMMENT ON COLUMN gold.v_kpi_summary.completed_sessions IS 'Total count of completed charging sessions.';
COMMENT ON COLUMN gold.v_kpi_summary.failed_sessions IS 'Total count of failed charging sessions.';
COMMENT ON COLUMN gold.v_kpi_summary.success_rate_pct IS 'Percentage of sessions that completed successfully.';
COMMENT ON COLUMN gold.v_kpi_summary.total_energy_kwh IS 'Total energy delivered across all completed sessions in kWh.';
COMMENT ON COLUMN gold.v_kpi_summary.total_billed IS 'Total amount billed to members for completed sessions.';
COMMENT ON COLUMN gold.v_kpi_summary.total_paid IS 'Total amount actually paid by members.';
COMMENT ON COLUMN gold.v_kpi_summary.revenue_leakage IS 'Difference between total billed and total paid (revenue leakage).';
COMMENT ON COLUMN gold.v_kpi_summary.mismatch_sessions IS 'Count of sessions where billed amount differs from amount paid.';
COMMENT ON COLUMN gold.v_kpi_summary.active_members IS 'Count of distinct members with at least one session.';